# memopro Colab T4 통합 시험 ① 언어 모델 (docs/research/0197, E042)

Qwen2.5-3B·7B의 생성과 16비트 LoRA를 그냥 GPU에 올리기 대 memopro(호스트 메모리에서 GPU로 흘려 쓰기)로 잰다. 7B는 T4(15GB)에 그냥은 안 들어간다.

- 예상 시간: 약 40~70분(첫 실행은 모델 다운로드 약 21GB 포함)
- 판정: G(생성 동일), L(LoRA 손실이 예산과 무관하게 비트 동일)

**나눈 노트북 네 개**(①언어 모델 ②비전 ③속도 ④데이터)는 Drive의 **같은 결과 폴더**(`results/suite/`)를 쓴다. 아무 순서로 하나씩 실행하면 되고, 끝난 경우는 건너뛴다. 동시에 두 세션에서 돌리지 않는다. `summary.md`는 지금까지 끝난 부분을 모두 판정하고, 아직 안 돌린 부분은 `n/a (not run yet)`로 나온다.

**준비**: 런타임 → 런타임 유형 변경 → **T4 GPU**. Drive `memopro_colab/install/`에 소스 묶음 `memopro-src.tar.gz`(첫 실행 때 빌드 5~8분). 이전 세션에서 memopro를 설치했다면 **런타임을 다시 시작**한다.

**준비 (처음 한 번, 세 노트북 공통)**
1. 런타임 → 런타임 유형 변경 → **T4 GPU**.
2. Google Drive에 `memopro_colab/install/` 폴더를 만들고 memopro 소스 배포본 `memopro-0.1.0a1.tar.gz`를 올린다(Linux wheel이 있으면 그것을 쓴다). 소스 배포본은 첫 실행 때 Rust로 빌드한다(3~5분).
   - 또는 Colab 왼쪽 열쇠(보안 비밀)에 `GITHUB_TOKEN`을 **직접** 등록하면 저장소에서 설치한다.
3. 셀을 실행한다. Drive 연결 허용 창이 뜨면 허용한다. 설정은 셀 맨 위(`SETTINGS`)에서 바꾼다.

**결과**: `MyDrive/memopro_colab/results/suite/<실행 id>/`
- `summary.md`·`summary.csv`: 비교 표와 읽는 법
- `cases/*.json`: 경우마다 모든 측정값(메모리, 시간, memopro 보고, 오류와 추적)
- `logs/*.log`: 경우마다 프로세스 출력 전체
- `timelines/*.csv`: 0.5초 간격 GPU·호스트 메모리와 GPU 사용률
- `env.json`: GPU, 판, memopro 빌드 확인
- 같은 이름의 `.zip`: 폴더 전체(공유용)

**끊겼을 때**: 같은 셀을 다시 실행하면 끝난 경우는 건너뛰고 이어서 한다(`NEW_RUN = True`면 새로 시작). 모델은 Drive의 `hf_cache/`에 남아 다시 받지 않는다. 세 노트북은 같은 Drive 폴더와 모델 캐시를 함께 쓴다.

**공정성**: 모든 경우는 **새 프로세스**에서 돈다(앞 경우의 메모리가 섞이지 않게, 0055). 여러 모델을 **동시에** 한 GPU에서 재면 서로의 측정을 오염시키므로(0085), 여러 모델은 목록으로 차례로 잰다.


In [ ]:
# ==== SETTINGS (edit here) ===========================================================
# memopro Colab T4 suite (E042, pre-registered in docs/research/0197): language models, vision
# models and ordinary data programs in one cell, each case in a fresh process, results on Drive.
#   LM   Qwen2.5 3B / 7B: greedy generation and 16-bit LoRA, the model loaded normally on the
#        T4 vs streamed from host memory to the GPU (stream_model(device="cuda"), 0195)
#   VIS  ResNet-152 (CNN) and DINOv2-giant (1.1 B parameters): inference, DINOv2 LoRA
#   DATA E027 image stack, E040 pandas / scikit-learn / simulation: unchanged, then under the
#        memopro-preload library at half their memory (Linux userfaultfd)
# Disk: the models (~26 GB) are kept in the Drive cache (MODEL_CACHE). About 2-3 hours.
LM_MODELS = {  # model -> two host budgets for streaming (bytes)
    "Qwen/Qwen2.5-3B-Instruct": [2 << 30, 3 << 30],
    "Qwen/Qwen2.5-7B-Instruct": [4 << 30, 6 << 30],
}
GEN_TOKENS = 32
LORA = {"steps": 5, "seq": 512}
VISION_MODELS = {  # model -> two host budgets (bytes), about 1/4 and 1/2 of the weights
    "microsoft/resnet-152": [60 << 20, 120 << 20],
    "facebook/dinov2-giant": [1100 << 20, 2200 << 20],
}
VISION_BATCH = 8
VISION_LORA = {"model": "facebook/dinov2-giant", "steps": 5, "batch": 8}
DATA_WORKLOADS = ["image.py", "dataframe.py", "classify.py", "simulate.py"]
DATA_CHUNK = 1 << 20  # pager chunk bytes (memopro-preload)
TIMEOUT_S = 60 * 60
# Which parts this notebook runs. All parts share one run folder on Drive, so the split notebooks
# (colab_t4_suite_*.ipynb) can be run one after another in separate sessions; finished cases are
# skipped and summary.md covers every part done so far.
PARTS = ["lm"]
MODEL_CACHE = "drive"  # "drive": keep downloaded models in Drive hf_cache/; "local": this session only
# ==== (settings shared by all cells) ================================================
DRIVE_ROOT = "/content/drive/MyDrive/memopro_colab"
GIT_REF = "main"
EXPECTED_COMMIT = "553224e"
NEW_RUN = False  # False: resume the last run of this cell if its settings did not change
RETRY_FAILED = True  # on resume, run crashed/timed-out cases again
STAGE_TO_LOCAL = True  # copy models from the Drive cache to the local disk before loading
# ==== END OF SETTINGS ================================================================
# ======================================================================================
# memopro Colab T4 heavy validation: shared bootstrap (embedded in every cell by build.py)
# Drive layout (DRIVE_ROOT, default /content/drive/MyDrive/memopro_colab):
#   install/        put memopro-src.tar.gz (source bundle: memopro + memopro-preload, 0197),
#                   or memopro-*.whl / memopro-*.tar.gz (sdist) here
#   hf_cache/       Hugging Face cache (models and datasets persist across sessions)
#   results/<cell>/<run_id>/   env.json, config.json, cases/*.json, logs/*.log,
#                              timelines/*.csv, summary.md, summary.csv, progress.jsonl
# ======================================================================================
import csv
import datetime
import glob
import json
import os
import shutil
import subprocess
import sys
import threading
import time

IN_COLAB = "google.colab" in sys.modules or os.path.isdir("/content")
LOCAL_SMOKE = os.environ.get("MP_LOCAL_SMOKE") == "1"  # local check of the cell logic, no Colab


LOG_FILE = None  # set by bootstrap: the cell's own progress log inside the run folder (0096)


def _log(msg):
    line = f"[{datetime.datetime.now():%H:%M:%S}] {msg}"
    print(line, flush=True)
    if LOG_FILE:
        with open(LOG_FILE, "a") as f:
            f.write(line + "\n")


def mount_drive():
    if LOCAL_SMOKE:
        root = os.environ["MP_DRIVE_ROOT"]
    else:
        from google.colab import drive  # noqa: PLC0415

        if not os.path.isdir("/content/drive/MyDrive"):
            drive.mount("/content/drive")
        root = DRIVE_ROOT
    for sub in ("install", "hf_cache", "results"):
        os.makedirs(os.path.join(root, sub), exist_ok=True)
    return root


def _pip(*args):
    return subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args],
                          capture_output=True, text=True, check=False)


def _rust():
    cargo = os.path.expanduser("~/.cargo/bin")
    if not os.path.exists(os.path.join(cargo, "cargo")):
        subprocess.run("curl -sSf https://sh.rustup.rs | sh -s -- -y --profile minimal",
                       shell=True, check=True, capture_output=True)
    os.environ["PATH"] = cargo + os.pathsep + os.environ["PATH"]


def _build_source(src):
    """memopro from a source tree (Rust core via maturin) and the Linux memopro-preload library."""
    _rust()
    _pip("maturin>=1.9,<2.0")
    r = _pip("--force-reinstall", "--no-deps", src)
    info = {"ok": r.returncode == 0, "stderr": r.stderr[-1500:] if r.returncode else ""}
    b = subprocess.run(["cargo", "build", "-q", "--release", "-p", "memopro-preload"], cwd=src,
                       capture_output=True, text=True, check=False)
    lib = os.path.join(src, "target", "release", "libmemopro_preload.so")
    if b.returncode == 0 and os.path.exists(lib):
        os.environ["MP_PRELOAD_LIB"] = lib
        info["preload"] = lib
    else:
        info["preload_error"] = b.stderr[-1000:]
    return info


def install_memopro(root):
    """source bundle in install/ (memopro + memopro-preload) > wheel > sdist (builds the Rust
    core) > GitHub with a Colab secret."""
    if LOCAL_SMOKE:
        import memopro  # noqa: PLC0415

        return {"source": "local environment", "version": memopro.__version__}
    try:
        import memopro  # noqa: PLC0415

        if os.environ.get("MP_MEMOPRO_INSTALLED") == "1":
            return {"source": "already installed in this session", "version": memopro.__version__}
    except ImportError:
        pass
    info = {}
    bundles = sorted(glob.glob(os.path.join(root, "install", "memopro-src*.tar.gz")))
    wheels = sorted(glob.glob(os.path.join(root, "install", "memopro-*linux*.whl")))
    sdists = sorted(glob.glob(os.path.join(root, "install", "memopro-*.tar.gz")))
    if bundles:
        _log("building memopro and memopro-preload from the source bundle (~5-8 min the first time)")
        src = "/content/memopro-src"
        shutil.rmtree(src, ignore_errors=True)
        os.makedirs(src)
        subprocess.run(["tar", "-xzf", bundles[-1], "-C", src], check=True)
        info = {"source": "source bundle", "file": os.path.basename(bundles[-1]),
                **_build_source(src)}
    elif wheels:
        r = _pip("--force-reinstall", "--no-deps", wheels[-1])
        info = {"source": "wheel", "file": os.path.basename(wheels[-1]), "ok": r.returncode == 0}
    elif sdists:
        _log("building memopro from the sdist (installs a Rust toolchain once, ~3-5 min)")
        _rust()
        _pip("maturin>=1.9,<2.0")
        r = _pip("--force-reinstall", "--no-deps", sdists[-1])
        info = {"source": "sdist", "file": os.path.basename(sdists[-1]), "ok": r.returncode == 0,
                "stderr": r.stderr[-1500:] if r.returncode else ""}
    else:
        token = None
        try:
            from google.colab import userdata  # noqa: PLC0415

            token = userdata.get("GITHUB_TOKEN")
        except Exception:  # noqa: BLE001
            token = None
        if not token:
            raise RuntimeError(
                "no memopro package: put the sdist (memopro-*.tar.gz) or a Linux wheel in "
                f"{os.path.join(root, 'install')}, or add a GITHUB_TOKEN Colab secret"
            )
        src = "/content/memopro-src"
        shutil.rmtree(src, ignore_errors=True)
        c = subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", GIT_REF,
                            f"https://{token}@github.com/imhyensuk/memopro.git", src],
                           capture_output=True, text=True, check=False)
        info = {"source": f"github@{GIT_REF}", "ok": False,
                "stderr": c.stderr[-800:].replace(token, "***")}
        if c.returncode == 0:
            info.update(_build_source(src))
            info["stderr"] = info.get("stderr", "").replace(token, "***")
    if not info.get("ok"):
        raise RuntimeError(f"memopro install failed: {info}")
    os.environ["MP_MEMOPRO_INSTALLED"] = "1"
    return info


def install_extras():
    if LOCAL_SMOKE:
        return {}
    want = ["bitsandbytes", "accelerate", "peft", "datasets"]
    missing = []
    for name in want:
        try:
            __import__(name)
        except ImportError:
            missing.append(name)
    if missing:
        _pip(*missing)
    removed = []
    try:  # Colab ships torchao 0.10, which peft >= 0.21 refuses when it adds any LoRA layer (0200)
        from importlib.metadata import version  # noqa: PLC0415

        major, minor = (int(x) for x in version("torchao").split(".")[:2])
        if (major, minor) < (0, 16):
            subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"],
                           capture_output=True, check=False)
            removed.append(f"torchao {major}.{minor}")
    except Exception:  # noqa: BLE001 - not installed, or an unusual version string
        pass
    return {"installed": missing, "removed": removed}


def environment(install_info):
    import platform  # noqa: PLC0415

    env = {"time": datetime.datetime.now().isoformat(timespec="seconds"),
           "python": sys.version.split()[0], "platform": platform.platform(),
           "install": install_info, "expected_commit": EXPECTED_COMMIT}
    for mod in ("memopro", "torch", "transformers", "accelerate", "bitsandbytes", "peft",
                "datasets", "huggingface_hub"):
        try:
            env[mod] = __import__(mod).__version__
        except Exception as e:  # noqa: BLE001
            env[mod] = f"unavailable: {type(e).__name__}"
    try:
        import torch  # noqa: PLC0415

        env["cuda"] = torch.cuda.is_available()
        if env["cuda"]:
            p = torch.cuda.get_device_properties(0)
            env["gpu"] = {"name": p.name, "total_bytes": p.total_memory,
                          "capability": f"{p.major}.{p.minor}"}
    except Exception as e:  # noqa: BLE001
        env["torch_error"] = str(e)
    try:
        with open("/proc/meminfo") as f:
            env["host_total_bytes"] = int(f.readline().split()[1]) * 1024
    except OSError:
        pass
    env["local_disk_free_bytes"] = shutil.disk_usage("/content" if IN_COLAB and not LOCAL_SMOKE
                                                     else os.getcwd()).free
    try:  # memopro build contents: which decisions this build includes
        import memopro.access._load as L  # noqa: PLC0415
        import memopro.orchestrator.candidates as C  # noqa: PLC0415

        env["build_has"] = {"0085_quality_note": hasattr(C, "quality_note"),
                            "0081_mps_heap_note": hasattr(L, "_suggest_mps_heap_setting")}
        import memopro._run as R  # noqa: PLC0415

        env["build_has"]["0089_elastic_default"] = hasattr(R, "default_elastic")
        env["build_has"]["0094_fixes"] = hasattr(C, "SLOW_BF16_NOTE") and not R.default_elastic()
        import memopro.access._train as T  # noqa: PLC0415

        env["build_has"]["0099_fixes"] = hasattr(C, "speed_hint") and hasattr(T, "even_micro")
        env["build_has"]["0100_fixes"] = hasattr(T, "optimizer_state_to_come")
        env["build_has"]["0103_fixes"] = hasattr(T, "release_cuda_cache")
        import memopro.rt.torch as RT  # noqa: PLC0415

        env["build_has"]["0195_cuda_stream"] = hasattr(RT, "_renamed")
        env["preload_lib"] = os.environ.get("MP_PRELOAD_LIB")
    except Exception as e:  # noqa: BLE001
        env["build_has"] = f"unknown: {e}"
    return env


# ------------------------------------------------------------------ run directory, resume
class Run:
    def __init__(self, root, cell, config, new_run=False):
        base = os.path.join(root, "results", cell)
        os.makedirs(base, exist_ok=True)
        latest = os.path.join(base, "LATEST")
        rid = None
        if not new_run and os.path.exists(latest):
            rid = open(latest).read().strip()
            prev = os.path.join(base, rid, "config.json")
            if not os.path.exists(prev) or json.load(open(prev)) != config:
                _log("config changed since the last run: starting a new run")
                rid = None
        if rid is None:
            rid = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
            with open(latest, "w") as f:
                f.write(rid)
        self.dir = os.path.join(base, rid)
        for sub in ("cases", "logs", "timelines", "reports"):
            os.makedirs(os.path.join(self.dir, sub), exist_ok=True)
        with open(os.path.join(self.dir, "config.json"), "w") as f:
            json.dump(config, f, indent=1)
        self.id, self.cell = rid, cell
        _log(f"results: {self.dir}")

    def path(self, *parts):
        return os.path.join(self.dir, *parts)

    def done(self, key):
        p = self.path("cases", key + ".json")
        if not os.path.exists(p):
            return None
        rec = json.load(open(p))
        if RETRY_FAILED and rec.get("status") in ("crashed", "error"):
            return None  # try a crashed or failed case again on resume (0200); a timeout is a
            # result (it re-ran a known 1-hour timeout in every session, 0220)
        return rec

    def save(self, key, rec):
        with open(self.path("cases", key + ".json"), "w") as f:
            json.dump(rec, f, indent=1, default=str)
        with open(self.path("progress.jsonl"), "a") as f:
            f.write(json.dumps({"t": time.time(), "case": key, "status": rec.get("status"),
                                "seconds": rec.get("wall_s")}) + "\n")

    def records(self):
        out = []
        for p in sorted(glob.glob(self.path("cases", "*.json"))):
            out.append(json.load(open(p)))
        return out


# ------------------------------------------------------------------ system timeline sampler
class Timeline:
    """nvidia-smi and host memory every `period` seconds while a case runs (CSV per case)."""

    def __init__(self, path, period=0.5):
        self.path, self.period, self.stop = path, period, threading.Event()
        self.peak_gpu_used, self.peak_host_used = 0, 0
        self.thread = threading.Thread(target=self._run, daemon=True)
        self.thread.start()

    def _gpu(self):
        try:
            out = subprocess.run(["nvidia-smi", "--query-gpu=memory.used,utilization.gpu",
                                  "--format=csv,noheader,nounits"], capture_output=True,
                                 text=True, timeout=5, check=False).stdout.strip().split(",")
            return int(out[0]) * 2**20, int(out[1])
        except Exception:  # noqa: BLE001
            return None, None

    def _host(self):
        try:
            info = {}
            with open("/proc/meminfo") as f:
                for line in f:
                    k, v = line.split(":", 1)
                    info[k] = int(v.split()[0]) * 1024
            return info["MemTotal"] - info["MemAvailable"], info.get("SwapTotal", 0) - info.get("SwapFree", 0)
        except Exception:  # noqa: BLE001
            return None, None

    def _run(self):
        t0 = time.time()
        with open(self.path, "w", newline="") as f:
            w = csv.writer(f)
            w.writerow(["t_s", "gpu_used_bytes", "gpu_util_pct", "host_used_bytes", "swap_used_bytes"])
            while not self.stop.is_set():
                g, u = self._gpu()
                h, s = self._host()
                w.writerow([round(time.time() - t0, 2), g, u, h, s])
                f.flush()
                if g:
                    self.peak_gpu_used = max(self.peak_gpu_used, g)
                if h:
                    self.peak_host_used = max(self.peak_host_used, h)
                self.stop.wait(self.period)

    def close(self):
        self.stop.set()
        self.thread.join(timeout=5)
        return {"timeline_peak_gpu_used_bytes": self.peak_gpu_used or None,
                "timeline_peak_host_used_bytes": self.peak_host_used or None}


# ------------------------------------------------------------------ workers
WORK_DIR = "/content/memopro_colab" if IN_COLAB and not LOCAL_SMOKE else os.path.join(
    os.environ.get("MP_DRIVE_ROOT", "."), "work")


def write_workers():
    os.makedirs(WORK_DIR, exist_ok=True)
    for name, source in WORKERS.items():
        with open(os.path.join(WORK_DIR, name), "w") as f:
            f.write(source)


def run_case(run, key, worker, args, timeout, env_extra=None):
    """One case in a fresh process; stdout/stderr to logs/<key>.log; result JSON to cases/."""
    prev = run.done(key)
    if prev is not None:
        _log(f"skip {key}: already {prev.get('status')} (resume)")
        return prev
    out_json = run.path("cases", key + ".tmp.json")
    log_path = run.path("logs", key + ".log")
    env = dict(os.environ)
    env.update({"HF_HOME": HF_HOME, "PYTHONUNBUFFERED": "1", "MP_OUT": out_json,
                "TOKENIZERS_PARALLELISM": "false"})
    env.update(env_extra or {})
    cmd = [sys.executable, os.path.join(WORK_DIR, worker), *[str(a) for a in args]]
    _log(f"run {key}")
    tl = Timeline(run.path("timelines", key + ".csv"))
    t0 = time.time()
    status = None
    with open(log_path, "w") as log:
        log.write("$ " + " ".join(cmd) + "\n")
        log.flush()
        try:
            proc = subprocess.run(cmd, stdout=log, stderr=subprocess.STDOUT, env=env,
                                  timeout=timeout, check=False)
            status = None if proc.returncode == 0 else f"crashed (exit {proc.returncode})"
        except subprocess.TimeoutExpired:
            status = "timeout"
    extra = tl.close()
    rec = {}
    if os.path.exists(out_json):
        rec = json.load(open(out_json))
        os.remove(out_json)
    if status and not rec.get("status"):
        rec["status"] = "timeout" if status == "timeout" else "crashed"
        rec["exit"] = status
        with open(log_path) as f:
            rec["log_tail"] = f.read()[-3000:]
    rec.setdefault("status", "ok")
    rec.update({"case": key, "wall_s": round(time.time() - t0, 1), **extra})
    run.save(key, rec)
    _log(f"  -> {rec['status']} ({rec['wall_s']}s)")
    return rec


# ------------------------------------------------------------------ models on Drive / local disk
def fetch_model(repo):
    """Download once into the Drive cache; copy to local disk for fast loading when it fits."""
    if LOCAL_SMOKE:
        return repo
    from huggingface_hub import snapshot_download  # noqa: PLC0415

    t = time.time()
    path = snapshot_download(repo, cache_dir=os.path.join(HF_HOME, "hub"),
                             allow_patterns=["*.json", "*.safetensors", "*.model", "*.txt",
                                             "tokenizer*", "*.tiktoken"])
    _log(f"model {repo} on Drive ({time.time() - t:.0f}s)")
    if not STAGE_TO_LOCAL:
        return path
    size = sum(os.path.getsize(os.path.realpath(p)) for p in glob.glob(os.path.join(path, "*")))
    local = os.path.join("/content/models", local_name(repo))
    if os.path.isdir(local):
        return local
    free = shutil.disk_usage("/content").free
    if free < size + 10 * 2**30:
        _log(f"not enough local disk to stage {repo} ({size / 2**30:.1f} GiB + 10 GiB margin, "
             f"{free / 2**30:.1f} GiB free); loading from Drive")
        return path
    t = time.time()
    partial = local + ".partial"  # a copy cut short (disconnect) must not look complete (0100)
    shutil.rmtree(partial, ignore_errors=True)
    shutil.copytree(path, partial, symlinks=False)
    os.rename(partial, local)
    _log(f"staged {repo} to local disk ({size / 2**30:.1f} GiB, {time.time() - t:.0f}s)")
    return local


def local_name(repo):
    return repo.replace("/", "--")


def free_local_models(keep=()):
    for d in glob.glob("/content/models/*"):
        if os.path.basename(d) not in keep:
            shutil.rmtree(d, ignore_errors=True)


# ------------------------------------------------------------------ summaries
def fmt_bytes(n):
    return "-" if not isinstance(n, (int, float)) else f"{n / 2**30:.2f} GiB"


def fmt(v, digits=2):
    if v is None:
        return "-"
    if isinstance(v, float):
        return f"{v:.{digits}f}"
    return str(v)


def write_summary(run, title, columns, rows, notes=()):
    """columns: list of (header, function(record) -> value)."""
    with open(run.path("summary.csv"), "w", newline="") as f:
        w = csv.writer(f)
        w.writerow([h for h, _ in columns])
        for r in rows:
            w.writerow([fn(r) for _, fn in columns])
    lines = [f"# {title}", "", f"run `{run.id}`, cell `{run.cell}`", ""]
    lines.append("| " + " | ".join(h for h, _ in columns) + " |")
    lines.append("|" + "---|" * len(columns))
    for r in rows:
        lines.append("| " + " | ".join(str(fn(r)) for _, fn in columns) + " |")
    lines += ["", *notes]
    text = "\n".join(lines) + "\n"
    with open(run.path("summary.md"), "w") as f:
        f.write(text)
    archive = shutil.make_archive(run.dir, "zip", run.dir)
    _log(f"summary: {run.path('summary.md')}\narchive to share: {archive}")
    print(text)
    return text


def bootstrap(cell, config):
    root = mount_drive()
    global HF_HOME
    HF_HOME = os.environ.get("MP_HF_HOME") or os.path.join(root, "hf_cache")
    os.environ["HF_HOME"] = HF_HOME
    info = install_memopro(root)
    extras = install_extras()
    write_workers()
    run = Run(root, cell, config, new_run=NEW_RUN)
    global LOG_FILE
    LOG_FILE = run.path("orchestrator.log")
    env = environment({**info, "extras": extras})
    with open(run.path("env.json"), "w") as f:
        json.dump(env, f, indent=1, default=str)
    _log(json.dumps({k: env.get(k) for k in ("memopro", "torch", "transformers", "gpu",
                                                "build_has")}, default=str))
    return run, env


WORKERS = {
    'worker_common.py': r'''"""Shared helpers for the Colab workers (each case runs in a fresh process)."""

import gc
import json
import os
import resource
import sys
import threading
import time
import traceback

import torch

DEVICE = os.environ.get("MP_DEVICE") or (
    "cuda" if torch.cuda.is_available()
    else "mps" if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available()
    else "cpu"
)
RESULT: dict = {"status": None, "device": DEVICE, "argv": sys.argv[1:]}
_T0 = time.time()


def sync():
    if DEVICE == "cuda":
        torch.cuda.synchronize()
    elif DEVICE == "mps":
        torch.mps.synchronize()


def peak_rss():
    """This process's peak resident memory. On Linux from /proc/self/status (VmHWM): getrusage's
    ru_maxrss survives execve, so a worker started from a large notebook kernel reported the
    kernel's peak (5.19 GiB in every case of E042's first run, 0199)."""
    try:
        with open("/proc/self/status") as f:
            for line in f:
                if line.startswith("VmHWM:"):
                    return int(line.split()[1]) * 1024
    except OSError:
        pass
    rss = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
    return rss if sys.platform == "darwin" else rss * 1024


class DeviceMonitor:
    """Peak device memory in use (cudaMemGetInfo: includes the CUDA context and other processes'
    use, i.e. what nvidia-smi shows) sampled every 20 ms, plus the allocator's own peaks."""

    def __init__(self):
        self.peak_used = 0
        self._stop = threading.Event()
        if DEVICE == "cuda":
            threading.Thread(target=self._run, daemon=True).start()

    def _run(self):
        while not self._stop.is_set():
            free, total = torch.cuda.mem_get_info()
            self.peak_used = max(self.peak_used, total - free)
            self._stop.wait(0.02)

    def reset(self):
        self.peak_used = 0
        if DEVICE == "cuda":
            sync()
            torch.cuda.reset_peak_memory_stats()

    def snapshot(self):
        out = {"host_peak_rss_bytes": peak_rss()}
        if DEVICE == "cuda":
            sync()
            free, total = torch.cuda.mem_get_info()
            out.update(allocated_bytes=torch.cuda.memory_allocated(),
                       reserved_bytes=torch.cuda.memory_reserved(),
                       peak_allocated_bytes=torch.cuda.max_memory_allocated(),
                       peak_reserved_bytes=torch.cuda.max_memory_reserved(),
                       device_used_bytes=total - free,
                       peak_device_used_bytes=max(self.peak_used, total - free))
        elif DEVICE == "mps":
            out.update(allocated_bytes=torch.mps.current_allocated_memory(),
                       driver_bytes=torch.mps.driver_allocated_memory())
        return out


MON = DeviceMonitor()


def is_oom(e):
    text = f"{type(e).__name__} {e}".lower()
    return isinstance(e, torch.OutOfMemoryError) or "out of memory" in text


def cleanup():
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
    elif DEVICE == "mps":
        torch.mps.empty_cache()


def memopro_report():
    try:
        import memopro

        return [{"technique": e.technique, "action": e.action, "detail": e.detail}
                for e in memopro.report().entries]
    except Exception as e:  # noqa: BLE001
        return [{"error": str(e)}]


def finish(status=None, **fields):
    RESULT.update(fields)
    if status:
        RESULT["status"] = status
    RESULT.setdefault("status", "ok")
    if RESULT["status"] is None:
        RESULT["status"] = "ok"
    RESULT["worker_s"] = round(time.time() - _T0, 1)
    RESULT["memory_end"] = MON.snapshot()
    RESULT["memopro_report"] = memopro_report()
    path = os.environ.get("MP_OUT")
    text = json.dumps(RESULT, default=str)
    if path:
        with open(path, "w") as f:
            f.write(text)
    print("RESULT " + text[:2000], flush=True)


def guarded(main):
    """Run main(); record an OOM or any other error as the case's result instead of crashing."""
    try:
        main()
    except Exception as e:  # noqa: BLE001
        status = "oom" if is_oom(e) else "error"
        finish(status, error=f"{type(e).__name__}: {e}"[:1500], trace=traceback.format_exc()[-3000:])
        return
    if RESULT.get("status") is None:
        finish()


def wikitext_ids(tokenizer, split, n_tokens):
    """WikiText-2 raw tokens (the revision pinned in 0017), joined with blank lines."""
    from datasets import load_dataset

    ds = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1",
                      revision="b08601e04326c79dfdd32d625aee71d232d685c3", split=split)
    text = "\n\n".join(t for t in ds["text"][: 20000 if split == "train" else None])
    ids = tokenizer(text, return_tensors="pt", verbose=False).input_ids[0]
    if ids.numel() < n_tokens:
        reps = n_tokens // ids.numel() + 1
        ids = ids.repeat(reps)
    return ids[:n_tokens]


def model_path(model_id):
    return os.environ.get("MP_MODEL_PATH") or model_id
''',
    'worker_train.py': r'''"""Training worker: one (model, scenario) per process. Same data, seed and steps in every
scenario, so loss curves and final parameters can be compared across scenarios."""

import argparse
import math
import statistics
import time

import torch
from worker_common import (
    DEVICE, MON, RESULT, cleanup, finish, guarded, is_oom, model_path, sync, wikitext_ids,
)

p = argparse.ArgumentParser()
p.add_argument("--model", required=True)
p.add_argument("--scenario", required=True)
p.add_argument("--batch", type=int, default=8)
p.add_argument("--seq", type=int, default=512)
p.add_argument("--steps", type=int, default=20)
p.add_argument("--cap", type=float, default=0.0, help="fraction of the GPU (emulates a smaller one)")
p.add_argument("--optimizer", default="adamw")
p.add_argument("--lr", type=float, default=1e-5)
A = p.parse_args()
RESULT.update(model=A.model, scenario=A.scenario, batch=A.batch, seq=A.seq, steps=A.steps,
              cap=A.cap, optimizer=A.optimizer)


def total_device():
    return torch.cuda.get_device_properties(0).total_memory if DEVICE == "cuda" else None


def apply_cap():
    if A.cap and DEVICE == "cuda":
        torch.cuda.set_per_process_memory_fraction(A.cap)
        RESULT["cap_bytes"] = int(A.cap * total_device())


def load_fp32():
    from transformers import AutoModelForCausalLM

    t = time.time()
    model = AutoModelForCausalLM.from_pretrained(model_path(A.model), dtype=torch.float32)
    for m in model.modules():  # no dropout: every scenario sees the same computation
        if isinstance(m, torch.nn.Dropout):
            m.p = 0.0
    model.config.use_cache = False
    model.to(DEVICE)
    RESULT["load_s"] = round(time.time() - t, 1)
    RESULT["n_params"] = sum(q.numel() for q in model.parameters())
    return model


def batches(vocab_limit=None):
    from transformers import AutoTokenizer

    tok = AutoTokenizer.from_pretrained(model_path(A.model))
    ids = wikitext_ids(tok, "train", A.steps * A.batch * A.seq)
    return ids.view(A.steps, A.batch, A.seq)


def make_opt(params):
    params = [q for q in params if q.requires_grad]
    if A.optimizer == "sgd":
        return torch.optim.SGD(params, lr=A.lr)
    return torch.optim.AdamW(params, lr=A.lr)


def cuda_state():
    """What memopro's budget sees when a session starts (0099: free + cached, model held)."""
    if DEVICE != "cuda":
        return None
    sync()
    free, total = torch.cuda.mem_get_info()
    return {"free_bytes": free, "total_bytes": total, "allocated_bytes": torch.cuda.memory_allocated(),
            "reserved_bytes": torch.cuda.memory_reserved()}


def fingerprint(model):
    """Per-parameter L2 norms: equal fingerprints (to ~1e-6) mean equal training results."""
    return [float(q.detach().cpu().double().norm()) for q in model.parameters()
            if q.requires_grad]


def record_steps(losses, times):
    steady = times[2:] if len(times) > 3 else times
    med = statistics.median(steady) if steady else None
    RESULT.update(losses=losses, step_s=[round(t, 4) for t in times], steps_done=len(losses),
                  median_step_s=med,
                  tokens_per_s=(RESULT.get("effective_batch", A.batch) * A.seq / med) if med else None,
                  final_loss=losses[-1] if losses else None)


def loop(model, step_fn, data):
    losses, times = [], []
    for i in range(len(data)):
        sync()
        t = time.time()
        try:
            loss = step_fn(data[i].to(DEVICE))
        except Exception as e:  # noqa: BLE001
            if is_oom(e):
                RESULT["oom_at_step"] = i
                record_steps(losses, times)
                raise
            raise
        sync()
        times.append(time.time() - t)
        losses.append(float(loss))
        if i == 0:
            RESULT["memory_after_first_step"] = MON.snapshot()
    record_steps(losses, times)


# ------------------------------------------------------------------ scenarios
def scenario_check():
    import memopro

    t = time.time()
    r = memopro.check(model_path(A.model), goal="train", batch_size=A.batch, seq_len=A.seq,
                      optimizer=A.optimizer, device=DEVICE,
                      **({"budget": int(A.cap * total_device())} if A.cap and DEVICE == "cuda" else {}))
    RESULT["check_s"] = round(time.time() - t, 2)
    RESULT["check"] = r.to_json()
    RESULT["predicted_peak"] = r.training.get("peak")
    RESULT["check_says_fits"] = r.training.get("fits")
    # the same work, measured: fp32, a steady-state (second) step
    apply_cap()
    try:
        model = load_fp32()
        opt = make_opt(model.parameters())
        x = batches()[0].to(DEVICE)
        for step in range(2):
            if step == 1:
                MON.reset()
            model(input_ids=x, labels=x).loss.backward()
            opt.step()
            opt.zero_grad(set_to_none=True)
        snap = MON.snapshot()
        measured = snap.get("peak_allocated_bytes")
        RESULT["measured_peak"] = measured
        pred = RESULT["predicted_peak"]
        RESULT["rel_error"] = (pred / measured - 1) if pred and measured else None
    except Exception as e:  # noqa: BLE001
        if not is_oom(e):
            raise
        RESULT["measured_peak"] = "oom"
        RESULT["check_correct_about_fit"] = RESULT["check_says_fits"] is False


def scenario_plain(amp=False, ckpt=False):
    apply_cap()
    model = load_fp32()
    if ckpt:
        model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    model.train()
    opt = make_opt(model.parameters())
    data = batches()
    scaler = torch.amp.GradScaler(DEVICE) if amp and DEVICE in ("cuda", "mps") else None
    MON.reset()

    def step(x):
        if amp:
            with torch.autocast(DEVICE, dtype=torch.float16):
                loss = model(input_ids=x, labels=x).loss
        else:
            loss = model(input_ids=x, labels=x).loss
        if scaler:
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
        else:
            loss.backward()
            opt.step()
        opt.zero_grad(set_to_none=True)
        return loss.detach()

    loop(model, step, data)
    RESULT["fingerprint"] = fingerprint(model)
    RESULT["memory"] = MON.snapshot()


def scenario_accel():
    from accelerate.utils import find_executable_batch_size

    apply_cap()
    model = load_fp32()
    model.train()
    opt = make_opt(model.parameters())
    data = batches()
    tries = []
    MON.reset()

    @find_executable_batch_size(starting_batch_size=A.batch)
    def train(batch_size):
        tries.append(batch_size)
        RESULT["effective_batch"] = batch_size

        def step(x):
            x = x[:batch_size]
            loss = model(input_ids=x, labels=x).loss
            loss.backward()
            opt.step()
            opt.zero_grad(set_to_none=True)
            return loss.detach()

        loop(model, step, data)

    train()
    RESULT["tries"] = tries
    RESULT["note"] = "accelerate shrinks the batch: the effective batch and the result change"
    RESULT["fingerprint"] = fingerprint(model)
    RESULT["memory"] = MON.snapshot()


def scenario_memopro(quality=None, micro=None):
    import memopro

    apply_cap()
    model = load_fp32()
    model.train()
    opt = make_opt(model.parameters())
    data = batches()
    kw = {}
    if A.cap and DEVICE == "cuda":
        kw["budget"] = int(A.cap * total_device())  # memopro would measure a device this size
    if quality:
        kw["quality"] = quality
    if micro:
        kw["micro_batch_size"] = micro  # a fixed split: the exact reference for the planned one
    MON.reset()
    trace = []
    RESULT["session_start"] = cuda_state()
    with memopro.train_session(model, opt, **kw) as s:
        RESULT["session_measured"] = cuda_state()  # after memopro returned the cache (0103)
        def step(x):
            loss = s.step({"input_ids": x}, lambda mb: model(input_ids=mb["input_ids"],
                                                             labels=mb["input_ids"]).loss)
            trace.append({"micro": s.micro, "active": s.active(), "retries": s.retries})
            return loss

        loop(model, step, data)
        RESULT["session"] = {"micro": s.micro, "active": s.active(), "retries": s.retries}
    RESULT["session_trace"] = trace
    RESULT["fingerprint"] = fingerprint(model)
    RESULT["memory"] = MON.snapshot()


def _lora(model):
    from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

    model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
    cfg = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.0, task_type="CAUSAL_LM",
                     target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj",
                                     "up_proj", "down_proj"])
    model = get_peft_model(model, cfg)
    RESULT["trainable_params"] = sum(q.numel() for q in model.parameters() if q.requires_grad)
    return model


def scenario_qlora(use_memopro):
    if DEVICE != "cuda":
        finish("skipped", reason="QLoRA needs CUDA (bitsandbytes)")
        return
    import memopro
    from transformers import AutoModelForCausalLM, BitsAndBytesConfig

    t = time.time()
    if use_memopro:
        model = memopro.load(model_path(A.model), device="cuda", quality="low")
    else:
        q = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                               bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
        model = AutoModelForCausalLM.from_pretrained(model_path(A.model), quantization_config=q,
                                                     device_map={"": 0}, dtype=torch.float16)
    RESULT["load_s"] = round(time.time() - t, 1)
    RESULT["layers"] = sorted({type(m).__name__ for m in model.modules() if "Linear" in type(m).__name__})
    model.config.use_cache = False
    model = _lora(model)
    model.train()
    opt = make_opt(model.parameters())
    data = batches()
    MON.reset()
    if use_memopro:
        RESULT["session_start"] = cuda_state()
        trace = []
        with memopro.train_session(model, opt) as s:
            RESULT["session_measured"] = cuda_state()  # after memopro returned the cache (0103)
            def step(x):
                loss = s.step({"input_ids": x}, lambda mb: model(input_ids=mb["input_ids"],
                                                                 labels=mb["input_ids"]).loss)
                trace.append({"micro": s.micro, "active": s.active(), "retries": s.retries})
                return loss

            loop(model, step, data)
            RESULT["session"] = {"micro": s.micro, "active": s.active(), "retries": s.retries}
        RESULT["session_trace"] = trace
    else:
        def step(x):
            loss = model(input_ids=x, labels=x).loss
            loss.backward()
            opt.step()
            opt.zero_grad(set_to_none=True)
            return loss.detach()

        loop(model, step, data)
    RESULT["memory"] = MON.snapshot()


def main():
    torch.manual_seed(0)
    s = A.scenario
    if s == "check":
        scenario_check()
    elif s == "plain":
        scenario_plain()
    elif s == "plain_amp":
        scenario_plain(amp=True)
    elif s == "hf_ckpt":
        scenario_plain(ckpt=True)
    elif s == "accel_find_batch":
        scenario_accel()
    elif s == "memopro":
        scenario_memopro()
    elif s == "memopro_lossless":
        scenario_memopro("lossless")
    elif s == "memopro_micro1":
        scenario_memopro(micro=1)
    elif s == "qlora_hf":
        scenario_qlora(False)
    elif s == "qlora_memopro":
        scenario_qlora(True)
    else:
        raise ValueError(f"unknown scenario {s}")
    losses = RESULT.get("losses") or []
    if losses and not all(math.isfinite(x) for x in losses):
        RESULT["warning"] = "non-finite loss"
    cleanup()


guarded(main)
''',
    'worker_infer.py': r'''"""Inference worker: one (model, variant) per process. Loads the model the variant's way, then
measures memory, time to first token for long prompts, decoding speed, WikiText-2 perplexity and
greedy outputs (token ids, for agreement across variants)."""

import argparse
import math
import time

import torch
from worker_common import (
    DEVICE, MON, RESULT, cleanup, finish, guarded, model_path, sync, wikitext_ids,
)

p = argparse.ArgumentParser()
p.add_argument("--model", required=True)
p.add_argument("--variant", required=True)
p.add_argument("--prompt-lens", default="512,2048")
p.add_argument("--new-tokens", type=int, default=128)
p.add_argument("--ppl-windows", type=int, default=8)
p.add_argument("--ppl-seq", type=int, default=1024)
p.add_argument("--offload-dir", default="/content/offload")
A = p.parse_args()
RESULT.update(model=A.model, variant=A.variant)

PROMPTS = [
    "Explain in three sentences why a GPU can run out of memory during training.",
    "Write a short Python function that returns the n-th Fibonacci number.",
    "What are the trade-offs of 4-bit quantization for large language models?",
]


def load():
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

    path = model_path(A.model)
    tok = AutoTokenizer.from_pretrained(path)
    v = A.variant
    MON.reset()
    t = time.time()
    if v.startswith("memopro"):
        import memopro

        quality = {"memopro": None, "memopro_low": "low", "memopro_high": "high"}[v]
        from memopro.access._load import plan_load

        plan = plan_load(path, device=DEVICE, **({"quality": quality} if quality else {}))
        RESULT["plan"] = [{"name": c.name, "device": c.needs.device, "host": c.needs.host,
                           "disk": c.needs.disk} for c in plan.candidates]
        try:
            model = memopro.load(path, device=DEVICE, **({"quality": quality} if quality else {}))
        except memopro.BudgetExceeded as e:
            RESULT["suggestions"] = [str(s) for s in getattr(e, "suggestions", [])]
            finish("does_not_fit", error=str(e)[:1500])
            return None, tok
        applied = [e for e in memopro.report().entries if e.action == "applied"]
        RESULT["memopro_choice"] = applied[-1].technique if applied else None
        RESULT["memopro_detail"] = applied[-1].detail if applied else None
    else:
        kw = {"device_map": "auto", "offload_folder": A.offload_dir}
        if v == "hf_fp16_auto":
            kw["dtype"] = torch.float16
        elif v == "hf_bnb8":
            kw["quantization_config"] = BitsAndBytesConfig(load_in_8bit=True)
        elif v == "hf_bnb4":
            kw["quantization_config"] = BitsAndBytesConfig(
                load_in_4bit=True, bnb_4bit_quant_type="nf4",
                bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
        else:
            raise ValueError(v)
        if DEVICE != "cuda":
            kw = {"dtype": torch.float16}
        model = AutoModelForCausalLM.from_pretrained(path, **kw)
        if DEVICE != "cuda":
            model.to(DEVICE)
    sync()
    RESULT["load_s"] = round(time.time() - t, 1)
    RESULT["memory_after_load"] = MON.snapshot()
    placement = {}
    for q in list(model.parameters()) + list(model.buffers()):
        d = q.device.type
        placement[d] = placement.get(d, 0) + q.numel() * q.element_size()
    RESULT["weights_by_device_bytes"] = placement
    dm = getattr(model, "hf_device_map", None)
    if dm:
        counts = {}
        for where in dm.values():
            counts[str(where)] = counts.get(str(where), 0) + 1
        RESULT["hf_device_map_modules"] = counts
    model.eval()
    return model, tok


def first_device(model):
    return next(model.parameters()).device if DEVICE != "cuda" else torch.device("cuda")


@torch.no_grad()
def measure(model, tok):
    dev = first_device(model)
    corpus = wikitext_ids(tok, "test", max(int(x) for x in A.prompt_lens.split(",")) + 64)
    # time to first token for long prompts
    ttft = {}
    for n in (int(x) for x in A.prompt_lens.split(",")):
        ids = corpus[:n].unsqueeze(0).to(dev)
        model.generate(ids[:, :16], max_new_tokens=2, do_sample=False)  # warm-up
        sync()
        t = time.time()
        model.generate(ids, max_new_tokens=1, do_sample=False)
        sync()
        ttft[n] = round(time.time() - t, 3)
        RESULT["ttft_s"] = ttft
    # decoding speed after a short prompt
    ids = corpus[:64].unsqueeze(0).to(dev)
    sync()
    t = time.time()
    model.generate(ids, max_new_tokens=1, do_sample=False)
    sync()
    t1 = time.time() - t
    t = time.time()
    out = model.generate(ids, max_new_tokens=A.new_tokens, min_new_tokens=A.new_tokens,
                         do_sample=False)
    sync()
    total = time.time() - t
    RESULT["decode_tok_s"] = (A.new_tokens - 1) / max(total - t1, 1e-6)
    RESULT["decode_total_s"] = round(total, 2)
    RESULT["generated_new"] = int(out.shape[1] - ids.shape[1])
    # perplexity on WikiText-2 test windows
    need = A.ppl_windows * A.ppl_seq
    ppl_ids = wikitext_ids(tok, "test", need).view(A.ppl_windows, A.ppl_seq)
    nll, n_tok = 0.0, 0
    for w in range(A.ppl_windows):
        x = ppl_ids[w : w + 1].to(dev)
        loss = model(input_ids=x, labels=x).loss
        nll += float(loss) * (A.ppl_seq - 1)
        n_tok += A.ppl_seq - 1
    RESULT["ppl"] = math.exp(nll / n_tok)
    RESULT["ppl_tokens"] = n_tok
    # greedy answers (token ids for agreement across variants, text for reading)
    answers = []
    for prompt in PROMPTS:
        msgs = [{"role": "user", "content": prompt}]
        if getattr(tok, "chat_template", None):
            enc = tok.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt",
                                          return_dict=True)
        else:
            enc = tok(prompt, return_tensors="pt")
        enc = {k: v.to(dev) for k, v in enc.items()}
        o = model.generate(**enc, max_new_tokens=48, do_sample=False,
                           pad_token_id=tok.pad_token_id or tok.eos_token_id)
        new = o[0][enc["input_ids"].shape[1]:]
        answers.append({"ids": new.tolist(), "text": tok.decode(new, skip_special_tokens=True)})
    RESULT["answers"] = answers
    RESULT["memory"] = MON.snapshot()


def check_prediction():
    """memopro.check's inference prediction for the stored model (reported next to measurements)."""
    try:
        import memopro

        r = memopro.check(model_path(A.model), goal="infer", batch_size=1, seq_len=2048,
                          device=DEVICE)
        RESULT["check_infer"] = r.to_json()
    except Exception as e:  # noqa: BLE001
        RESULT["check_infer"] = {"error": f"{type(e).__name__}: {e}"[:300]}


def main():
    torch.manual_seed(0)
    if A.variant == "memopro":
        check_prediction()
    model, tok = load()
    if model is None:
        return
    measure(model, tok)
    del model
    cleanup()


guarded(main)
''',
    'worker_multi.py': r'''"""Multi-model and process-level worker: doctor accuracy, several models taking turns on one GPU
(keep all / delete and reload / memopro hibernate), and regression checks of recent changes."""

import argparse
import json
import os
import time

import torch
from worker_common import DEVICE, MON, RESULT, cleanup, finish, guarded, is_oom, sync

p = argparse.ArgumentParser()
p.add_argument("--scenario", required=True)
p.add_argument("--models", default="", help="comma-separated local paths (rotation)")
p.add_argument("--labels", default="")
p.add_argument("--strategy", default="reload")
p.add_argument("--rounds", type=int, default=2)
p.add_argument("--new-tokens", type=int, default=32)
p.add_argument("--small-model", default="")
p.add_argument("--spill-dir", default="/content/spill")
A = p.parse_args()
RESULT.update(scenario=A.scenario, strategy=A.strategy)
PROMPT = "List three reasons why a GPU can run out of memory."


def scenario_doctor():
    import memopro

    t = time.time()
    d = memopro.doctor()
    RESULT["doctor_s"] = round(time.time() - t, 2)
    RESULT["summary"] = d.summary()
    RESULT["warnings"] = d.warnings()
    budget = d.budget
    RESULT["memopro_device_budget"] = budget.device
    RESULT["memopro_host_budget"] = budget.host
    if DEVICE == "cuda":
        free, total = torch.cuda.mem_get_info()
        RESULT["cuda_free"], RESULT["cuda_total"] = free, total
        RESULT["device_budget_over_free"] = budget.device / free if budget.device else None
    try:
        with open("/proc/meminfo") as f:
            info = {line.split(":")[0]: int(line.split()[1]) * 1024 for line in f}
    except OSError:  # not Linux
        info = {}
    RESULT["host_mem_available"] = info.get("MemAvailable")
    RESULT["host_budget_over_available"] = (
        budget.host / info["MemAvailable"] if budget.host and info.get("MemAvailable") else None)


def _load(path):
    from transformers import AutoModelForCausalLM, AutoTokenizer

    model = AutoModelForCausalLM.from_pretrained(path, dtype=torch.float16).to(DEVICE).eval()
    return model, AutoTokenizer.from_pretrained(path)


@torch.no_grad()
def _answer(model, tok):
    msgs = [{"role": "user", "content": PROMPT}]
    if getattr(tok, "chat_template", None):
        enc = tok.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt",
                                      return_dict=True).to(DEVICE)
    else:
        enc = tok(PROMPT, return_tensors="pt").to(DEVICE)
    out = model.generate(**enc, max_new_tokens=A.new_tokens, do_sample=False,
                         pad_token_id=tok.pad_token_id or tok.eos_token_id)
    return out[0][enc["input_ids"].shape[1]:].tolist()


def scenario_rotate():
    """Models take turns: A, B, C, A, B, C, ... Each turn answers the same prompt."""
    import memopro

    paths = A.models.split(",")
    labels = A.labels.split(",") if A.labels else paths
    if A.strategy == "hibernate":
        os.makedirs(A.spill_dir, exist_ok=True)
        memopro.configure(disk_writes="allow", spill_dir=A.spill_dir)
    loaded, handles, first_answer, turns = {}, {}, {}, []
    MON.reset()
    t_all = time.time()
    for r in range(A.rounds):
        for i, path in enumerate(paths):
            turn = {"round": r, "model": labels[i]}
            sync()
            t = time.time()
            if A.strategy == "reload":
                for k in list(loaded):
                    del loaded[k]
                cleanup()
                loaded[i] = _load(path)
            elif A.strategy == "keep_all":
                if i not in loaded:
                    loaded[i] = _load(path)
            elif A.strategy == "hibernate":
                for k, h in list(handles.items()):  # put the others to sleep
                    if k != i and h is None:
                        handles[k] = memopro.hibernate.now(loaded[k][0], name=labels[k])
                        turn.setdefault("slept", []).append(
                            {"model": labels[k], "modes": dict(handles[k].bytes_by_mode()),
                             "disk_write_bytes": handles[k].disk_write_bytes})
                cleanup()
                if i in handles and handles[i] is not None:
                    handles[i].wake()
                    handles[i] = None
                elif i not in loaded:
                    loaded[i] = _load(path)
                    handles[i] = None
            sync()
            turn["ready_s"] = round(time.time() - t, 2)
            t = time.time()
            ids = _answer(*loaded[i])  # no local reference: reload must be able to free it (0093)
            sync()
            turn["answer_s"] = round(time.time() - t, 2)
            if i in first_answer:
                turn["same_answer_as_first_turn"] = ids == first_answer[i]
            else:
                first_answer[i] = ids
            turn["memory"] = MON.snapshot()
            turns.append(turn)
            RESULT["turns"] = turns
    RESULT["total_s"] = round(time.time() - t_all, 1)
    RESULT["all_answers_repeat"] = all(t.get("same_answer_as_first_turn", True) for t in turns)
    RESULT["memory"] = MON.snapshot()


def scenario_regress():
    """Recent changes (0059-0089) on CUDA/Linux: each check is independent."""
    import memopro

    checks = {}

    def check(name, fn):
        try:
            checks[name] = fn()
        except Exception as e:  # noqa: BLE001
            checks[name] = {"error": f"{type(e).__name__}: {e}"[:400]}

    from memopro.access._load import plan_load
    from memopro.env import macos_malloc_cache_on, mps_heap_reserve_on

    small = A.small_model

    def int4_group():
        from memopro.orchestrator import candidates

        ctx = plan_load(small, device=DEVICE).ctx
        return {"group": candidates._int4_group(ctx, 4), "expect": 64,
                "pass": candidates._int4_group(ctx, 4) == 64}

    def int4_quality_note():
        from memopro.orchestrator import candidates

        # the note for this back end (0099 D9); older builds had one note for all
        note = getattr(candidates, "BNB_INT4_QUALITY_NOTE", None) or candidates.INT4_QUALITY_NOTE

        plan = plan_load(small, device=DEVICE, quality="low")
        int4 = next(c for c in plan.candidates if c.name == "quant.int4")
        memopro.report().clear()
        model = memopro.load(small, device=DEVICE, quality="low",
                             budget=int4.needs.device + int4.needs.host + (64 << 20))
        entry = [e for e in memopro.report().entries if e.action == "applied"][-1]
        layers = sorted({type(m).__name__ for m in model.modules() if "Linear" in type(m).__name__})
        del model
        cleanup()
        return {"chosen": entry.technique, "note_in_report": note in entry.detail,
                "layers": layers, "pass": note in entry.detail
                and entry.technique == "load.quant.int4" and "Int4PackedLinear" not in layers}

    def residency_file_refused():
        try:
            memopro.load(small, device=DEVICE, residency="file")
        except memopro.ModeUnavailable as e:
            return {"refused": True, "reason": str(e)[:200], "pass": True}
        return {"refused": False, "pass": False}

    def platform_defaults():
        from memopro._run import default_elastic

        notes = memopro.doctor(devices=False).warnings()
        return {"mps_heap_note": mps_heap_reserve_on(), "malloc_note": macos_malloc_cache_on(),
                "elastic_default_in_run": default_elastic(),
                "pass": not mps_heap_reserve_on() and not macos_malloc_cache_on()
                and default_elastic() is False  # off everywhere since 0094 (E021 D4)
                and not any("MPS" in n or "MallocLargeCache" in n for n in notes)}

    def pressure_signal():
        from memopro import elastic

        try:
            return {"reading": elastic.current(), "pass": True}
        except memopro.ModeUnavailable as e:
            return {"unsupported": str(e)[:200], "pass": None}

    def hibernate_bit_exact():
        from transformers import AutoModelForCausalLM

        model = AutoModelForCausalLM.from_pretrained(small, dtype=torch.float32).to(DEVICE)
        before = [q.detach().clone() for q in model.parameters()]
        out = {}
        for mode in ("host", "compress"):
            try:
                h = memopro.hibernate.now(model, mode=mode)
            except memopro.ModeUnavailable as e:  # e.g. host on unified memory
                out[mode] = {"unavailable": str(e)[:160]}
                continue
            asleep = MON.snapshot().get("allocated_bytes")
            h.wake()
            same = all(torch.equal(a, b) for a, b in zip(before, list(model.parameters()), strict=True))
            out[mode] = {"bit_exact": same, "allocated_while_asleep": asleep}
        del model, before
        cleanup()
        ran = [v for v in out.values() if "bit_exact" in v]
        out["pass"] = bool(ran) and all(v["bit_exact"] for v in ran)
        return out

    def check_infer_prediction():
        r = memopro.check(small, goal="infer", batch_size=1, seq_len=1024, device=DEVICE)
        return {"inference": r.inference, "pass": r.inference.get("peak") is not None}

    for name, fn in (("int4_group_on_cuda", int4_group), ("int4_quality_note", int4_quality_note),
                     ("residency_file_refused", residency_file_refused),
                     ("platform_defaults", platform_defaults), ("pressure_signal", pressure_signal),
                     ("hibernate_bit_exact", hibernate_bit_exact),
                     ("check_infer_prediction", check_infer_prediction)):
        check(name, fn)
        print(name, json.dumps(checks[name], default=str)[:300], flush=True)
    RESULT["checks"] = checks
    RESULT["all_pass"] = all(isinstance(v, dict) and v.get("pass") in (True, None)
                             for v in checks.values())


def main():
    torch.manual_seed(0)
    if A.scenario == "doctor":
        scenario_doctor()
    elif A.scenario == "rotate":
        try:
            scenario_rotate()
        except Exception as e:  # noqa: BLE001
            if is_oom(e):
                finish("oom", error=f"{type(e).__name__}: {e}"[:600])
                return
            raise
    elif A.scenario == "regress":
        scenario_regress()
    else:
        raise ValueError(A.scenario)


guarded(main)
''',
    'app_naive_load.py': r'''"""An unmodified user script: loads a model the simplest way and answers one prompt.
Run with plain `python` and with `memopro run` (no code change)."""

import sys
import time

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

path = sys.argv[1]
t = time.time()
tok = AutoTokenizer.from_pretrained(path)
model = AutoModelForCausalLM.from_pretrained(path)
if torch.cuda.is_available() and next(model.parameters()).device.type == "cpu":
    model = model.to("cuda")
print(f"APP loaded in {time.time() - t:.1f}s on {next(model.parameters()).device}", flush=True)
msgs = [{"role": "user", "content": "In one sentence, why do GPUs run out of memory?"}]
device = next(model.parameters()).device
if getattr(tok, "chat_template", None):
    enc = tok.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt",
                                  return_dict=True).to(device)
else:
    enc = tok(msgs[0]["content"], return_tensors="pt").to(device)
t = time.time()
out = model.generate(**enc, max_new_tokens=32, do_sample=False,
                     pad_token_id=tok.pad_token_id or tok.eos_token_id)
print(f"APP generated in {time.time() - t:.1f}s:",
      tok.decode(out[0][enc["input_ids"].shape[1]:], skip_special_tokens=True), flush=True)
if torch.cuda.is_available():
    print(f"APP peak_allocated {torch.cuda.max_memory_allocated()}", flush=True)
''',
    'worker_suite.py': r'''"""E042 worker (docs/research/0197): one case of the Colab T4 suite in a fresh process.

    worker_suite.py lm_gen    <model> <plain|memopro> <budget_bytes> <new_tokens>
    worker_suite.py lm_lora   <model> <plain|memopro> <budget_bytes> <steps> <seq>
    worker_suite.py vis_infer <model> <plain|memopro> <budget_bytes> <batch>
    worker_suite.py vis_lora  <model> <plain|memopro> <budget_bytes> <steps> <batch>

`plain` loads the model normally onto the GPU; `memopro` streams it from host memory within
`budget_bytes` (memopro.rt.torch.stream_model(device="cuda"), 0195).
"""

import glob
import hashlib
import os
import sys
import time

import torch
from worker_common import DEVICE, MON, RESULT, cleanup, finish, guarded, model_path, sync

PROMPTS = [
    "Explain how a hash table handles collisions.",
    "Write a Python function that checks whether a string is a palindrome, with a short docstring.",
]
VISION_CLASSES = {"resnet": "ResNetForImageClassification", "dinov2": "Dinov2Model",
                  "vit": "ViTModel"}


def copy_stats(m):
    """Device copies, cache hits, prefetched weights and the cache size (0201), if streamed."""
    w = getattr(m, "memopro_weights", None)
    if w is None or not hasattr(w, "copy_stats"):
        return None
    return {**w.copy_stats, "gpu_budget": w.gpu_budget, "gpu_cache_bytes": w.gpu_cache_bytes}


def sha(t):
    return hashlib.sha256(t.detach().float().cpu().numpy().tobytes()).hexdigest()[:16]


def vision_class(model_id):
    import transformers

    for key, name in VISION_CLASSES.items():
        if key in model_id.lower():
            return getattr(transformers, name)
    return transformers.AutoModel


def model_files(path):
    """Name, size and first 16 bytes of each file in a model folder."""
    out = []
    for f in sorted(glob.glob(os.path.join(path, "*"))):
        with open(f, "rb") as fh:
            head = fh.read(16)
        out.append({"name": os.path.basename(f), "real": os.path.realpath(f),
                    "size": os.path.getsize(f), "head": head.hex()})
    return out


def load(kind, model_id, mode, budget):
    import transformers

    import memopro.rt.torch as rtt

    path = model_path(model_id)
    cls = transformers.AutoModelForCausalLM if kind == "lm" else vision_class(model_id)
    t = time.time()
    if mode == "plain":
        dtype = torch.bfloat16 if kind == "lm" else None
        m = cls.from_pretrained(path, dtype=dtype).to(DEVICE).eval()
    else:
        try:
            m = rtt.stream_model(path, budget=budget, device=DEVICE, model_class=cls)
        except Exception:
            RESULT["files"] = model_files(path)  # what the streamer saw (DINOv2, 0220)
            raise
    sync()
    RESULT["load_s"] = round(time.time() - t, 1)
    RESULT["memory_after_load"] = MON.snapshot()
    return m, path


def lora_layers(m, device):
    """PEFT adapters of a streamed model start on the meta device: put them on the GPU (as
    memopro.finetune does)."""
    import peft

    for layer in m.modules():
        if isinstance(layer, peft.tuners.lora.LoraLayer):
            for name in layer.active_adapters:
                if layer.lora_A[name].weight.device.type == "meta":
                    layer.lora_A[name].to_empty(device=device)
                    layer.lora_B[name].to_empty(device=device)
                    layer.reset_lora_parameters(name, True)


def lm_gen(model_id, mode, budget, new_tokens):
    import transformers

    import memopro

    m, path = load("lm", model_id, mode, budget)
    tok = transformers.AutoTokenizer.from_pretrained(path)
    texts, secs = [], []
    for p in PROMPTS:
        t = time.time()
        if mode == "plain":
            msgs = [{"role": "user", "content": p}]
            ids = tok.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt")
            ids = (ids["input_ids"] if hasattr(ids, "keys") else ids).to(DEVICE)
            with torch.no_grad():
                out = m.generate(input_ids=ids, max_new_tokens=new_tokens, do_sample=False,
                                 pad_token_id=tok.eos_token_id)
            texts.append(tok.decode(out[0, ids.shape[1]:], skip_special_tokens=True))
        else:
            texts.append(memopro.generate(m, p, tokenizer=tok, max_new_tokens=new_tokens))
        sync()
        secs.append(time.time() - t)
    finish(texts=texts, seconds=secs, s_per_token=sum(secs) / (new_tokens * len(PROMPTS)),
           rt=getattr(getattr(m, "memopro_runtime", None), "stats", lambda: None)(),
           copy_stats=copy_stats(m))


def lm_lora(model_id, mode, budget, steps, seq):
    import transformers

    import memopro
    import memopro.rt.torch as rtt
    from worker_common import wikitext_ids

    if mode == "memopro":
        m, path = load("lm", model_id, mode, budget)
        tok = transformers.AutoTokenizer.from_pretrained(path)
        ids = wikitext_ids(tok, "train", (seq - 1) * steps)
        texts = [tok.decode(ids[i * (seq - 1):(i + 1) * (seq - 1)]) for i in range(steps)]
        r = memopro.finetune(m, texts, tokenizer=tok, seq_len=seq)
        finish(losses=r.losses, loss_bits=[float.hex(x) for x in r.losses],
               step_s=r.step_seconds, tokens=r.tokens, rt=m.memopro_runtime.stats())
        return
    import peft

    m, path = load("lm", model_id, mode, budget)
    tok = transformers.AutoTokenizer.from_pretrained(path)
    ids = wikitext_ids(tok, "train", (seq - 1) * steps)
    texts = [tok.decode(ids[i * (seq - 1):(i + 1) * (seq - 1)]) for i in range(steps)]
    eos = tok.eos_token or ""
    all_ids = tok(eos.join(texts) + eos, return_tensors="pt").input_ids[0]
    torch.manual_seed(0)
    m = peft.get_peft_model(m, peft.LoraConfig(r=8, lora_alpha=16,
                                               target_modules=["q_proj", "k_proj", "v_proj", "o_proj"]))
    for p in m.parameters():
        if p.requires_grad:
            p.data = p.data.float()
    m.train()
    m.base_model.model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": True})
    m.base_model.model.enable_input_require_grads()
    opt = torch.optim.AdamW([p for p in m.parameters() if p.requires_grad], lr=2e-4)
    losses, step_s = [], []
    pieces = [all_ids[i:i + seq] for i in range(0, len(all_ids) - 1, seq)]
    for piece in [p for p in pieces if len(p) > 1]:
        t = time.time()
        loss = rtt.causal_lm_loss(m.base_model.model, piece.unsqueeze(0).to(DEVICE))
        loss.backward()
        opt.step()
        opt.zero_grad(set_to_none=True)
        sync()
        losses.append(float(loss.detach()))
        step_s.append(time.time() - t)
    finish(losses=losses, loss_bits=[float.hex(x) for x in losses], step_s=step_s)


def images(batch, size):
    g = torch.Generator().manual_seed(0)
    return torch.rand(batch, 3, size, size, generator=g)


def features(m, x):
    out = m(pixel_values=x)
    logits = getattr(out, "logits", None)
    return logits if logits is not None else out.last_hidden_state[:, 0]


def vis_infer(model_id, mode, budget, batch):
    m, _ = load("vision", model_id, mode, budget)
    x = images(batch, 224).to(DEVICE)
    with torch.no_grad():
        t = time.time()
        y = features(m, x)
        sync()
        first = time.time() - t
        t = time.time()
        y2 = features(m, x)
        sync()
    finish(output_sha=sha(y), same_twice=bool(torch.equal(y, y2)), shape=list(y.shape),
           first_s=first, second_s=time.time() - t,
           rt=getattr(getattr(m, "memopro_runtime", None), "stats", lambda: None)(),
           copy_stats=copy_stats(m))


def vis_lora(model_id, mode, budget, steps, batch):
    """LoRA on the attention projections + a fresh linear head (10 classes) over the class token;
    the same random images and labels at every step."""
    import peft

    import memopro.rt.torch as rtt

    m, _ = load("vision", model_id, mode, budget)
    names = {n.split(".")[-1] for n, mod in m.named_modules() if isinstance(mod, torch.nn.Linear)}
    targets = [t for t in ("query", "value", "q_proj", "v_proj") if t in names]
    torch.manual_seed(0)
    m = peft.get_peft_model(m, peft.LoraConfig(r=8, lora_alpha=16, target_modules=targets))
    lora_layers(m, DEVICE)
    for p in m.parameters():
        if p.requires_grad:
            p.data = p.data.float()
    torch.manual_seed(1)
    hidden = m.base_model.model.config.hidden_size
    head = torch.nn.Linear(hidden, 10).to(DEVICE)
    params = [p for p in m.parameters() if p.requires_grad] + list(head.parameters())
    opt = torch.optim.AdamW(params, lr=1e-3)
    x = images(batch, 224).to(DEVICE)
    y = torch.arange(batch, device=DEVICE) % 10
    streamed = hasattr(m.base_model.model, "memopro_weights")
    losses, step_s = [], []
    for _ in range(steps):
        t = time.time()
        ctx = rtt.saved_weights(m.base_model.model) if streamed else torch.enable_grad()
        with ctx:
            f = features(m.base_model.model, x).float()
            loss = torch.nn.functional.cross_entropy(head(f), y)
            loss.backward()
        opt.step()
        opt.zero_grad(set_to_none=True)
        sync()
        losses.append(float(loss.detach()))
        step_s.append(time.time() - t)
    finish(losses=losses, loss_bits=[float.hex(v) for v in losses], step_s=step_s,
           targets=targets,
           rt=m.base_model.model.memopro_runtime.stats() if streamed else None)


def main():
    kind, model_id, mode, budget, *rest = sys.argv[1:]
    RESULT.update(kind=kind, model=model_id, mode=mode, budget=int(budget))
    torch.manual_seed(0)
    if kind.endswith("_lora"):  # the same steps give the same bits (CUBLAS_WORKSPACE_CONFIG set)
        torch.use_deterministic_algorithms(True, warn_only=True)
    fn = {"lm_gen": lm_gen, "lm_lora": lm_lora, "vis_infer": vis_infer, "vis_lora": vis_lora}[kind]
    fn(model_id, mode, int(budget), *map(int, rest))
    cleanup()


if __name__ == "__main__":
    guarded(main)
''',
    'worker_data.py': r'''"""E042 worker (docs/research/0197): one data workload (E027 image stack, E040 pandas /
scikit-learn / simulation) in this process, unchanged; started with or without LD_PRELOAD by the
cell. Its JSON line becomes the case's result.

    worker_data.py <workload.py>
"""

import contextlib
import io
import json
import os
import runpy
import sys

script = os.path.join(os.path.dirname(os.path.abspath(__file__)), sys.argv[1])
sys.path.insert(0, os.path.dirname(script))
sys.argv = [script]
out = io.StringIO()
status, error = "ok", None
try:
    with contextlib.redirect_stdout(out):
        runpy.run_path(script, run_name="__main__")
except MemoryError as e:
    status, error = "oom", repr(e)
except Exception as e:  # noqa: BLE001
    status, error = "error", f"{type(e).__name__}: {e}"
result = None
for line in reversed(out.getvalue().strip().splitlines()):
    with contextlib.suppress(ValueError):
        result = json.loads(line)
        break
# the workloads take ru_maxrss, which keeps the notebook kernel's peak across execve on Linux;
# this process's own peak is VmHWM (0199)
hwm = None
with contextlib.suppress(OSError), open("/proc/self/status") as f:
    for line in f:
        if line.startswith("VmHWM:"):
            hwm = int(line.split()[1]) * 1024
if result is not None and hwm is not None:
    result["maxrss_rusage_bytes"] = result.get("maxrss_bytes")
    result["maxrss_bytes"] = hwm
rec = {"status": status, "error": error, "result": result,
       "preloaded": "memopro_preload" in os.environ.get("LD_PRELOAD", "")}
with open(os.environ["MP_OUT"], "w") as f:
    json.dump(rec, f)
print("RESULT " + json.dumps(rec)[:2000], flush=True)
''',
    'image.py': r'''"""E027 workload (docs/research/0122): an ordinary NumPy program, run unchanged.

A stack of synthetic 16-bit grayscale images (default 2,048 of 512x512 = 1 GiB), built one image
at a time, then four passes of plain whole-array NumPy: per-image means, pixels over a threshold
(a full-size boolean temporary), background subtraction in place, and final statistics with a
SHA-256 of the whole stack. Prints one JSON line.

    python workload.py [images]
"""

import hashlib
import json
import resource
import sys
import time

import numpy as np

H = W = 512


def main() -> None:
    n = int(sys.argv[1]) if len(sys.argv) > 1 else 2048
    t0 = time.perf_counter()
    rng = np.random.default_rng(0)
    y = np.arange(H, dtype=np.float64)[:, None]
    x = np.arange(W, dtype=np.float64)[None, :]
    stack = np.empty((n, H, W), dtype=np.uint16)
    for i in range(n):
        smooth = 2000 + 900 * np.sin(x / 41 + i / 57) * np.cos(y / 67 - i / 91)
        noise = rng.integers(0, 4, size=(H, W), dtype=np.uint16)
        stack[i] = smooth.astype(np.uint16) + noise
    built = time.perf_counter() - t0

    means = stack.mean(axis=(1, 2))
    counts = (stack > 2600).sum(axis=(1, 2))
    background = stack.min(axis=0)
    stack -= background
    total = int(stack.sum(dtype=np.uint64))
    peak = int(stack.max())
    digest = hashlib.sha256(stack).hexdigest()
    seconds = time.perf_counter() - t0

    maxrss = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
    print(
        json.dumps(
            {
                "images": n,
                "means_sha256": hashlib.sha256(means.tobytes()).hexdigest(),
                "counts_sha256": hashlib.sha256(counts.tobytes()).hexdigest(),
                "total": total,
                "max": peak,
                "stack_sha256": digest,
                "build_seconds": built,
                "seconds": seconds,
                "maxrss_bytes": maxrss * (1 if sys.platform == "darwin" else 1024),
            }
        )
    )


if __name__ == "__main__":
    main()
''',
    'common.py': r'''"""Shared by the E040 workloads: the peak RSS so far and one JSON line at the end."""

import hashlib
import json
import resource
import sys
import time

T0 = time.perf_counter()


def rss() -> int:
    r = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
    return r * (1 if sys.platform == "darwin" else 1024)


IMPORT_RSS = rss()  # imported right after the workload's own imports


def sha(*arrays) -> str:
    h = hashlib.sha256()
    for a in arrays:
        h.update(memoryview(a).cast("B"))
    return h.hexdigest()


def done(**result) -> None:
    result |= {"seconds": time.perf_counter() - T0, "import_rss_bytes": IMPORT_RSS,
               "maxrss_bytes": rss()}
    print(json.dumps(result))
''',
    'dataframe.py': r'''"""E040 W1 (docs/research/0181): an ordinary pandas program, run unchanged — a sales table of
16 M rows (customer, store, time, price, quantity), revenue per row, totals per store, the top
100 customers, and the table sorted by store and time."""

import numpy as np
import pandas as pd
from common import done, sha

N = 16_000_000
rng = np.random.default_rng(0)
df = pd.DataFrame(
    {
        "customer": rng.integers(0, 1_000_000, N),
        "store": rng.integers(0, 500, N).astype(np.int32),
        "time": np.sort(rng.integers(1_600_000_000, 1_700_000_000, N)),
        "price": rng.integers(100, 50_000, N) / 100,
        "qty": rng.integers(1, 10, N).astype(np.int16),
    }
)
df["revenue"] = df["price"] * df["qty"]
per_store = df.groupby("store")["revenue"].sum()
top = df.groupby("customer")["revenue"].sum().nlargest(100)
df = df.sort_values(["store", "time"], kind="stable")
done(
    per_store=sha(np.ascontiguousarray(per_store.to_numpy())),
    top=sha(np.ascontiguousarray(top.to_numpy()), np.ascontiguousarray(top.index.to_numpy())),
    sorted_revenue=sha(np.ascontiguousarray(df["revenue"].to_numpy())),
    total=float(df["revenue"].sum()),
)
''',
    'classify.py': r'''"""E040 W2 (docs/research/0181): an ordinary scikit-learn program, run unchanged — a linear
classifier (SGD, 5 epochs) on sparse TF-IDF-like features of 400 k documents x 200 k terms
(Zipf-distributed term ids, ~120 terms per document), labels from a hidden linear rule."""

import numpy as np
import scipy.sparse as sp
from common import done, sha
from sklearn.linear_model import SGDClassifier

DOCS, TERMS, PER_DOC = 400_000, 200_000, 120
rng = np.random.default_rng(0)
nnz = DOCS * PER_DOC
cols = (rng.zipf(1.3, nnz) - 1) % TERMS
rows = np.repeat(np.arange(DOCS), PER_DOC)
vals = np.log1p(rng.integers(1, 5, nnz)).astype(np.float64)
X = sp.csr_matrix((vals, (rows, cols)), shape=(DOCS, TERMS))
X.sum_duplicates()
idf = np.log(DOCS / (1 + np.bincount(X.indices, minlength=TERMS)))
X = X.multiply(idf).tocsr()
w = rng.standard_normal(TERMS)
y = (X @ w > np.median(X @ w)).astype(np.int8)
clf = SGDClassifier(max_iter=5, tol=None, random_state=0).fit(X, y)
done(coef=sha(np.ascontiguousarray(clf.coef_)), accuracy=float(clf.score(X, y)), nnz=int(X.nnz))
''',
    'simulate.py': r'''"""E040 W3 (docs/research/0181): an ordinary NumPy simulation, run unchanged — 2-D heat
diffusion on a 4096 x 4096 float32 grid (explicit scheme, 300 steps), keeping every 10th field
(30 snapshots) in memory, then per-cell time mean and maximum over the history."""

import numpy as np
from common import done, sha

N, STEPS, EVERY = 4096, 300, 10
y, x = np.mgrid[0:N, 0:N].astype(np.float32) / N
u = np.zeros((N, N), np.float32)
for cx, cy, s in ((0.3, 0.3, 0.05), (0.7, 0.6, 0.08), (0.5, 0.8, 0.03)):
    u += np.exp(-((x - cx) ** 2 + (y - cy) ** 2) / (2 * s * s)).astype(np.float32)
del x, y
history = np.empty((STEPS // EVERY, N, N), np.float32)
for step in range(STEPS):
    lap = -4 * u
    lap[1:] += u[:-1]
    lap[:-1] += u[1:]
    lap[:, 1:] += u[:, :-1]
    lap[:, :-1] += u[:, 1:]
    u += np.float32(0.2) * lap
    if step % EVERY == 0:
        history[step // EVERY] = u
mean = history.mean(axis=0)
peak = history.max(axis=0)
done(mean=sha(mean), peak=sha(peak), history=sha(history), final=sha(u))
''',
    'preload_smoke.c': r'''/* memopro-preload smoke test (0189): run with
 *   LD_PRELOAD=libmemopro_preload.so MEMOPRO_PRELOAD_BUDGET=67108864 \
 *   MEMOPRO_PRELOAD_REPORT=report.json ./smoke
 * 24 blocks of 8 MiB (3x the budget) through malloc, calloc, realloc and posix_memalign, each
 * filled with a compressible pattern and read back exactly; then small blocks. Prints "ok". */
#define _POSIX_C_SOURCE 200112L
#include <stdint.h>
#include <stdio.h>
#include <stdlib.h>
#include <malloc.h>

#define BLOCKS 24
#define WORDS ((8u << 20) / 4)

static uint32_t pattern(size_t b, size_t i) { return (uint32_t)(b * 7919u + i / 1024u); }

int main(void) {
    uint32_t *blocks[BLOCKS];
    for (size_t b = 0; b < BLOCKS; b++) {
        void *p = NULL;
        if (b % 4 == 0) p = malloc(WORDS * 4);
        else if (b % 4 == 1) {
            p = calloc(WORDS, 4);
            for (size_t i = 0; p && i < WORDS; i += 4096)
                if (((uint32_t *)p)[i]) { puts("calloc not zero"); return 1; }
        } else if (b % 4 == 2) {
            p = malloc(1 << 20);
            p = p ? realloc(p, WORDS * 4) : NULL;
        } else if (posix_memalign(&p, 4096, WORDS * 4) != 0) p = NULL;
        if (!p) { printf("allocation %zu failed\n", b); return 1; }
        if (malloc_usable_size(p) < WORDS * 4) { puts("usable size too small"); return 1; }
        blocks[b] = p;
        for (size_t i = 0; i < WORDS; i++) blocks[b][i] = pattern(b, i);
    }
    for (int pass = 0; pass < 2; pass++)
        for (size_t b = 0; b < BLOCKS; b++)
            for (size_t i = 0; i < WORDS; i++)
                if (blocks[b][i] != pattern(b, i)) {
                    printf("block %zu word %zu differs\n", b, i);
                    return 1;
                }
    blocks[0] = realloc(blocks[0], WORDS * 8);
    for (size_t i = 0; i < WORDS; i++)
        if (blocks[0][i] != pattern(0, i)) { puts("realloc lost data"); return 1; }
    for (size_t b = 0; b < BLOCKS; b++) free(blocks[b]);
    for (int i = 0; i < 100000; i++) free(malloc((size_t)(i % 4000) + 1));
    puts("ok");
    return 0;
}
''',
}

# ---------------------------------------------------------------- body
MIB = 1 << 20
DETERMINISTIC = {"CUBLAS_WORKSPACE_CONFIG": ":4096:8"}


def short(model):
    return model.split("/")[-1]


def suite_body():
    config = {"LM_MODELS": LM_MODELS, "GEN_TOKENS": GEN_TOKENS, "LORA": LORA,
              "VISION_MODELS": VISION_MODELS, "VISION_BATCH": VISION_BATCH,
              "VISION_LORA": VISION_LORA, "DATA_WORKLOADS": DATA_WORKLOADS,
              "DATA_CHUNK": DATA_CHUNK, "MODEL_CACHE": MODEL_CACHE}
    if MODEL_CACHE == "local":
        os.environ["MP_HF_HOME"] = "/content/hf_cache"
    run, env = bootstrap("suite", config)
    has = env.get("build_has") if isinstance(env.get("build_has"), dict) else {}
    if not has.get("0195_cuda_stream"):
        raise RuntimeError(
            f"this memopro build cannot stream to CUDA (build_has: {has}): put the new source "
            "bundle memopro-src.tar.gz in Drive memopro_colab/install/, restart, run again")
    # ---- LM
    for model, budgets in (LM_MODELS if "lm" in PARTS else {}).items():
        free_local_models(keep=(local_name(model),))
        path = fetch_model(model)
        extra = {"MP_MODEL_PATH": path}
        n = short(model)
        run_case(run, f"lm_gen__{n}__plain", "worker_suite.py",
                 ["lm_gen", model, "plain", 0, GEN_TOKENS], TIMEOUT_S, extra)
        for b in budgets:
            run_case(run, f"lm_gen__{n}__memopro_{b // MIB}", "worker_suite.py",
                     ["lm_gen", model, "memopro", b, GEN_TOKENS], TIMEOUT_S, extra)
        run_case(run, f"lm_lora__{n}__plain", "worker_suite.py",
                 ["lm_lora", model, "plain", 0, LORA["steps"], LORA["seq"]], TIMEOUT_S,
                 {**extra, **DETERMINISTIC})
        for b in budgets:
            run_case(run, f"lm_lora__{n}__memopro_{b // MIB}", "worker_suite.py",
                     ["lm_lora", model, "memopro", b, LORA["steps"], LORA["seq"]], TIMEOUT_S,
                     {**extra, **DETERMINISTIC})
    # ---- vision
    for model, budgets in (VISION_MODELS if "vision" in PARTS else {}).items():
        free_local_models(keep=(local_name(model),))
        path = fetch_model(model)
        extra = {"MP_MODEL_PATH": path}
        n = short(model)
        run_case(run, f"vis_infer__{n}__plain", "worker_suite.py",
                 ["vis_infer", model, "plain", 0, VISION_BATCH], TIMEOUT_S, extra)
        for b in budgets:
            run_case(run, f"vis_infer__{n}__memopro_{b // MIB}", "worker_suite.py",
                     ["vis_infer", model, "memopro", b, VISION_BATCH], TIMEOUT_S, extra)
        if model == VISION_LORA["model"]:
            args = [VISION_LORA["steps"], VISION_LORA["batch"]]
            run_case(run, f"vis_lora__{n}__plain", "worker_suite.py",
                     ["vis_lora", model, "plain", 0, *args], TIMEOUT_S, {**extra, **DETERMINISTIC})
            for b in budgets:
                run_case(run, f"vis_lora__{n}__memopro_{b // MIB}", "worker_suite.py",
                         ["vis_lora", model, "memopro", b, *args], TIMEOUT_S,
                         {**extra, **DETERMINISTIC})
    # ---- speed after the GPU cache and prefetch (E043, docs/research/0202): the same memopro
    # cases again under new keys, each model's largest budget
    speed = {**LM_MODELS, **{VISION_LORA["model"]: VISION_MODELS[VISION_LORA["model"]]}}
    for model, budgets in (speed if "speed" in PARTS else {}).items():
        free_local_models(keep=(local_name(model),))
        path = fetch_model(model)
        n, b = short(model), budgets[-1]
        if model in LM_MODELS:
            run_case(run, f"lm_gen__{n}__memopro_{b // MIB}__cache", "worker_suite.py",
                     ["lm_gen", model, "memopro", b, GEN_TOKENS], TIMEOUT_S, {"MP_MODEL_PATH": path})
        else:
            run_case(run, f"vis_infer__{n}__memopro_{b // MIB}__cache", "worker_suite.py",
                     ["vis_infer", model, "memopro", b, VISION_BATCH], TIMEOUT_S,
                     {"MP_MODEL_PATH": path})
    free_local_models()
    # ---- data programs (CPU), unchanged and under memopro-preload at half their memory
    lib = os.environ.get("MP_PRELOAD_LIB")
    probe = preload_probe(run, lib) if "data" in PARTS else (run.done("data__probe") or {})
    for w in (DATA_WORKLOADS if "data" in PARTS else []):
        n = w.removesuffix(".py")
        for mode in ("plain", "preload"):  # measured before VmHWM (0199): measure again
            old = run.done(f"data__{n}__{mode}")
            if old is not None and "maxrss_rusage_bytes" not in (old.get("result") or {}):
                os.remove(run.path("cases", f"data__{n}__{mode}.json"))
        plain = run_case(run, f"data__{n}__plain", "worker_data.py", [w], TIMEOUT_S,
                         {"MP_DEVICE": "cpu"})
        peak = ((plain.get("result") or {}).get("maxrss_bytes")) or 0
        if not (probe.get("ok") and peak):
            continue
        limit = peak // 2
        report = run.path("reports", f"data__{n}__preload.json")
        rec = run_case(run, f"data__{n}__preload", "worker_data.py", [w], TIMEOUT_S, {
            "MP_DEVICE": "cpu", "LD_PRELOAD": lib, "MEMOPRO_PRELOAD_BUDGET": str(limit),
            "MEMOPRO_PRELOAD_PROCESS": str(limit - 64 * MIB),
            "MEMOPRO_PRELOAD_CHUNK": str(DATA_CHUNK), "MEMOPRO_PRELOAD_REPORT": report,
            "MEMOPRO_PRELOAD_REPORT_EVERY": "5"})
        if os.path.exists(report) and "pager" not in rec:
            rec["pager"] = json.load(open(report))
            rec["limit"] = limit
            run.save(rec["case"], rec)
    summarize_suite(run, probe)


def preload_probe(run, lib):
    """Does userfaultfd work here? The memopro-preload smoke test (C) pushes 3x its budget."""
    key = "data__probe"
    prev = run.done(key)
    if prev is not None:
        return prev
    rec = {"case": key, "lib": lib}
    if not lib or not os.path.exists(lib):
        rec.update(status="skipped", ok=False, why="memopro-preload was not built")
    else:
        exe = os.path.join(WORK_DIR, "preload_smoke")
        cc = subprocess.run(["cc", "-std=c11", "-O1", "-o", exe,
                             os.path.join(WORK_DIR, "preload_smoke.c")],
                            capture_output=True, text=True, check=False)
        report = run.path("reports", "probe.json")
        p = subprocess.run([exe], capture_output=True, text=True, timeout=600, check=False,
                           env={**os.environ, "LD_PRELOAD": lib,
                                "MEMOPRO_PRELOAD_BUDGET": str(64 * MIB),
                                "MEMOPRO_PRELOAD_REPORT": report}) if cc.returncode == 0 else None
        ok = p is not None and p.returncode == 0 and p.stdout.strip().endswith("ok")
        rec.update(status="ok" if ok else "failed", ok=ok,
                   output=(cc.stderr + (p.stdout + p.stderr if p else ""))[-2000:],
                   report=json.load(open(report)) if os.path.exists(report) else None)
    run.save(key, rec)
    _log(f"preload probe: {rec['status']} {rec.get('why', '')}")
    return rec


def summarize_suite(run, probe):
    recs = {r.get("case"): r for r in run.records()}
    rows = []

    def check(name, criterion, result, detail, keys=()):
        if keys and not any(recs.get(k) for k in keys):
            result, detail = None, "not run yet (its notebook part has not been run)"
        rows.append({"check": name, "criterion": criterion,
                     "result": "pass" if result is True else ("fail" if result is False else "n/a"),
                     "detail": detail})

    def ok(r):
        return r.get("status") == "ok"

    for model, budgets in LM_MODELS.items():
        n = short(model)
        p = recs.get(f"lm_gen__{n}__plain", {})
        ms = [recs.get(f"lm_gen__{n}__memopro_{b // MIB}", {}) for b in budgets]
        done = all(ok(m) for m in ms)
        same = done and ms[0].get("texts") == ms[1].get("texts")
        if ok(p):
            check(f"G-{n}", "streamed generation text = the model loaded normally (both budgets)",
                  done and all(m.get("texts") == p.get("texts") for m in ms),
                  f"s/token plain {fmt(p.get('s_per_token'))}, memopro {[fmt(m.get('s_per_token')) for m in ms]}",
                  [f"lm_gen__{n}__memopro_{b // MIB}" for b in budgets])
        else:
            check(f"G-{n}", "plain does not fit the T4; streamed completes, same text at both budgets",
                  done and same, f"plain {p.get('status')}; memopro {[m.get('status') for m in ms]}, "
                  f"s/token {[fmt(m.get('s_per_token')) for m in ms]}",
                  [f"lm_gen__{n}__memopro_{b // MIB}" for b in budgets])
        p = recs.get(f"lm_lora__{n}__plain", {})
        ms = [recs.get(f"lm_lora__{n}__memopro_{b // MIB}", {}) for b in budgets]
        done = all(ok(m) and len(m.get("losses") or []) == LORA["steps"] for m in ms)
        check(f"L-{n}", "16-bit LoRA completes; losses bit-identical across budgets",
              done and ms[0].get("loss_bits") == ms[1].get("loss_bits"),
              f"plain {p.get('status')} {[round(x, 4) for x in p.get('losses') or []]}; memopro "
              f"{[round(x, 4) for x in ms[0].get('losses') or []]}, step s {[fmt(sum(m.get('step_s') or [0]) / max(1, len(m.get('step_s') or [])), 1) for m in ms]}",
              [f"lm_lora__{n}__memopro_{b // MIB}" for b in budgets])
    for model, budgets in VISION_MODELS.items():
        n = short(model)
        p = recs.get(f"vis_infer__{n}__plain", {})
        ms = [recs.get(f"vis_infer__{n}__memopro_{b // MIB}", {}) for b in budgets]
        check(f"V-{n}", "streamed inference output = the model loaded normally (both budgets)",
              ok(p) and all(ok(m) and m.get("output_sha") == p.get("output_sha") for m in ms),
              f"plain {p.get('status')} {fmt(p.get('second_s'))} s; memopro "
              f"{[(m.get('status'), fmt(m.get('second_s'))) for m in ms]}",
              [f"vis_infer__{n}__memopro_{b // MIB}" for b in budgets])
    n = short(VISION_LORA["model"])
    budgets = VISION_MODELS[VISION_LORA["model"]]
    ms = [recs.get(f"vis_lora__{n}__memopro_{b // MIB}", {}) for b in budgets]
    p = recs.get(f"vis_lora__{n}__plain", {})
    check(f"VL-{n}", "vision LoRA completes; losses bit-identical across budgets",
          all(ok(m) for m in ms) and ms[0].get("loss_bits") == ms[1].get("loss_bits"),
          f"plain {p.get('status')} {[round(x, 4) for x in p.get('losses') or []]}; memopro "
          f"{[round(x, 4) for x in ms[0].get('losses') or []]}",
          [f"vis_lora__{n}__memopro_{b // MIB}" for b in budgets])
    for model, budgets in {**LM_MODELS, **{VISION_LORA["model"]: VISION_MODELS[VISION_LORA["model"]]}}.items():
        n, b = short(model), budgets[-1]
        kind = "lm_gen" if model in LM_MODELS else "vis_infer"
        p = recs.get(f"{kind}__{n}__plain", {})
        old = recs.get(f"{kind}__{n}__memopro_{b // MIB}", {})
        new = recs.get(f"{kind}__{n}__memopro_{b // MIB}__cache", {})
        same = (new.get("texts") == p.get("texts")) if kind == "lm_gen" else (
            new.get("output_sha") == p.get("output_sha"))
        speed = "s_per_token" if kind == "lm_gen" else "second_s"
        a, z, ref = old.get(speed), new.get(speed), p.get(speed)
        limit = {"Qwen2.5-7B-Instruct": 2.0}.get(n, 2 * (ref or 0))
        check(f"S-{n}", "same output as plain; time <= the pre-registered limit (0202)",
              ok(new) and same and z is not None and z <= limit,
              f"plain {fmt(ref)}, no cache {fmt(a)}, cache {fmt(z)} (limit {fmt(limit)}); "
              f"copies {new.get('copy_stats') or '-'}",
              [f"{kind}__{n}__memopro_{b // MIB}__cache"])
    for w in DATA_WORKLOADS:
        n = w.removesuffix(".py")
        p, m = recs.get(f"data__{n}__plain", {}), recs.get(f"data__{n}__preload", {})
        if not recs.get(f"data__{n}__plain") and not recs.get(f"data__{n}__preload"):
            check(f"D-{n}", "unchanged program at half memory under memopro-preload", None,
                  "not run yet (its notebook part has not been run)")
            continue
        if not probe.get("ok"):
            check(f"D-{n}", "unchanged program at half memory under memopro-preload", None,
                  f"userfaultfd probe: {probe.get('status')} {probe.get('why', '')}")
            continue
        pr, mr = p.get("result") or {}, m.get("result") or {}
        keys = [k for k in pr if k not in ("seconds", "build_seconds", "import_rss_bytes",
                                            "maxrss_bytes", "maxrss_rusage_bytes")]
        same = bool(mr) and all(mr.get(k) == pr.get(k) for k in keys)
        limit = m.get("limit") or 0
        within = bool(mr) and mr.get("maxrss_bytes", 1 << 62) <= limit + 64 * MIB
        pager = m.get("pager") or {}
        check(f"D-{n}", "same result; process peak <= half of the plain peak (+64 MiB); no overruns",
              same and within and pager.get("overruns") == 0,
              f"peak {fmt_bytes(mr.get('maxrss_bytes'))} vs limit {fmt_bytes(limit)} (plain "
              f"{fmt_bytes(pr.get('maxrss_bytes'))}); {fmt(pr.get('seconds'), 1)} -> "
              f"{fmt(mr.get('seconds'), 1)} s; overruns {pager.get('overruns')}")
    notes = ["Pre-registered criteria: docs/research/0197.", "", "## All cases", "",
             "| case | status | load s | host peak RSS | GPU peak used |", "|---|---|---|---|---|"]
    for key in sorted(recs):
        r = recs[key]
        mem = r.get("memory_end") or {}
        notes.append(f"| {key} | {r.get('status')} | {fmt(r.get('load_s'), 1)} | "
                     f"{fmt_bytes(mem.get('host_peak_rss_bytes'))} | "
                     f"{fmt_bytes(r.get('timeline_peak_gpu_used_bytes'))} |")
    errors = [(k, r) for k, r in sorted(recs.items()) if r.get("status") not in ("ok", None)
              and r.get("status") != "skipped"]
    if errors:
        notes += ["", "## Errors", ""]
        for key, r in errors:
            text = (r.get("error") or r.get("exit") or "").replace("|", "/").replace("\n", " ")
            notes.append(f"- `{key}` ({r.get('status')}): {text[:400]}")
    cols = [("check", lambda r: r["check"]), ("criterion", lambda r: r["criterion"]),
            ("result", lambda r: r["result"]), ("detail", lambda r: r["detail"])]
    return write_summary(run, "memopro Colab T4 suite (E042)", cols, rows, notes)


suite_body()
